# Make Figure 7 (F21)

### Load necessary packages, create a spectral model grid, and define some functions

In [ ]:
from jax import config
config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import astropy.units as u
from chromatic import *
import arviz as az
import shone
from fleck.jax import ActiveStar
from bt_settl import get_interp_stellar_spectrum_3d

# Create custom binned bin edges
wl_min, wl_max = 0.7, 1.7
n_points = 10000

binned_bin_edges = np.geomspace(wl_min, wl_max, n_points + 1)

binned_wavelengths = (binned_bin_edges[:-1] + binned_bin_edges[1:]) / 2.0

print(f"Creating custom binned grid with {n_points} points from {wl_min} to {wl_max} µm")
print(f"Binned bin edges shape: {len(binned_bin_edges)}")
print(f"Binned wavelengths shape: {len(binned_wavelengths)}")

# Step 2: Create the interpolation grid using these custom bin edges
print("\nCreating BT-Settl 3D interpolation grid with custom binning...")
panchromatic_btsettl_grid = get_interp_stellar_spectrum_3d(binned_bin_edges)
panchromatic_wavelengths = binned_wavelengths
panchromatic_bin_edges = binned_bin_edges

print(f"\nWavelength range: {panchromatic_wavelengths[0]:.3f} - {panchromatic_wavelengths[-1]:.3f} µm")
print(f"Number of wavelength points: {len(panchromatic_wavelengths)}")
print("\nGrid creation complete!")

def get_BTSettl_spectrum_jax(T, 
                             metallicity=0, data_wave=None,
                             grid=panchromatic_btsettl_grid):
    """
    Get BT-Settl spectrum for given temperature, logg, and metallicity.
    
    Parameters:
    T : effective temperature (K)
    metallicity : metallicity ([M/H])
    grid : 3D interpolation function from get_interp_stellar_spectrum_3d
    
    Returns:
    flux_jax : flux array (erg/cm^2/s/Å)
    """
    
    gridspec = grid(
        jnp.array(T, dtype=jnp.float64),
        jnp.array(metallicity, dtype=jnp.float64)
    )    
    
    flux_jax = jnp.array(gridspec, dtype=jnp.float64)

    binned_flux = shone.bin_spectrum(data_wave, panchromatic_wavelengths, flux_jax)
    
    return data_wave, binned_flux

def breathing_model_jax(phase, b1, b2, b3, b4):

    phase = jnp.array(phase)
    _breathing = 1. + (b1 * phase) + (b2 * phase**2.) + (b3 * phase**3.) + (b4 * phase**4.)
    breathing = _breathing/jnp.mean(_breathing)
    
    return jnp.array(breathing)

def ramp_model_jax(phase, r1, r2, r3):

    phase = jnp.array(phase)
    _ramp = 1. - jnp.exp( (-r1 * phase) + r2) + (r3 * phase)
    ramp = _ramp/jnp.mean(_ramp)
    
    return jnp.array(ramp)

def linear_model_jax(x, m):

    x = jnp.array(x)
    _line = m * x + 1
    line = _line/jnp.mean(_line)

    return jnp.array(line)

visits = {
    'F21': {
            'Grism': 'G141',
            'BJD_times': np.array(pd.read_csv('../../data/F21_bjdtimes.csv')['BJD'][:]) * u.day,
            'time_lower': 2459455.708 * u.day,
            'time_upper': 2459455.738 * u.day,
            'T0 (BJD_TDB)': 2459455.9895 * u.day,
            'exp (s)': 4.9784 * u.s,
            'native resolution': 46.3 * u.angstrom
        },
    }

### Open the data file (stored as a chromatic "rainbow" object)

In [ ]:
visit = 'F21'
orbits_to_exclude = np.array([0,1,2,6,7]) # Only fitting the transit

# The way we're using fleck, it wants a multi-value wavelength array.
# So we define 3 bin edges, making 2 bins, and calling the second bin as the F21 wavelengths
lc_index = 1 if visit=='F21' else 0
bin_edges = jnp.array([0.791,1.135,1.64]) * u.micron

predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
binwidth = visits[f'{visit}']['native resolution']
exptime = visits[f'{visit}']['exp (s)']
grism = visits[f'{visit}']['Grism']

rainbow = read_rainbow(f"../../data/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
speclc_rainbow = rainbow

binned_for_model = rainbow.bin(wavelength_edges=bin_edges)
fleck_wavelengths = binned_for_model.wavelength.value

integrated_flux = np.trapz(speclc_rainbow.flux.value,x=speclc_rainbow.wavelength.value,axis=0)

img_date = speclc_rainbow.time.value
data_flux = integrated_flux/integrated_flux.mean()
relative_err = np.sqrt(integrated_flux/speclc_rainbow.nwave)/integrated_flux
time_from_T0 = img_date - predicted_T0

# plt.figure(figsize=(9,3))
# plt.errorbar(time_from_T0,data_flux,relative_err*data_flux,
#              label=f'Median Err = {int(np.nanmedian(relative_err)*1e6) }ppm',
#             fmt='o',ms=1)
# plt.legend()
# plt.title(f'{visit} Integrated Light Curve')
# plt.show()
# plt.clf()

### Label the orbits, trim the first point from each orbit and mask some orbits

In [ ]:
# Label the orbits
orbit = np.zeros_like(img_date)
for j in range(len(img_date)):
    if j >= 1:
        if (img_date[j] - img_date[j - 1]) > 0.01:
            orbit[j] = (orbit[j - 1] + 1)
        else:
            orbit[j] = orbit[j - 1]

# Trim the first point from each orbit
ref_time = []
for o in np.unique(orbit):
    first_index = np.where(orbit == o)[0][0]
    ref_time.append(img_date[first_index])
    data_flux[first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
    relative_err[first_index] = np.nan
    img_date[first_index] = np.nan
    time_from_T0[first_index] = np.nan

time_from_T0[94:116] = np.nan

# Set data to nan if it was in the pre-defined list of orbits to exclude
for orbit_to_exclude in orbits_to_exclude:
    data_flux[orbit == orbit_to_exclude] = np.nan
    relative_err[orbit == orbit_to_exclude] = np.nan
    img_date[orbit == orbit_to_exclude] = np.nan
    time_from_T0[orbit == orbit_to_exclude] = np.nan

# plt.figure(figsize=(9,3) )
# plt.errorbar(time_from_T0,data_flux,yerr=relative_err,fmt='o',ms=1)
# plt.show()
# plt.clf()

# Populate ramp_phase time arrays
phase_list=[]
for o in [0,1,2,3,4,5,6,7]:
    rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
    phase_list.append(rphase)
ramp_phase = np.concatenate(phase_list)
breathing_phase = ( (img_date-ref_time[0]+0.02) / 0.066 ) % 1

### Exclude all nan values and make the arrays compatible with JAX. Not strictly necessary here, but necessary for the modeling.

In [ ]:
# Convert all arrays to JAX arrays
_data_flux = jnp.array( data_flux[~np.isnan(time_from_T0)] )
data_flux = _data_flux[~jnp.isnan(_data_flux)]
_relative_err = jnp.array(relative_err[~np.isnan(time_from_T0)] )
relative_err = _relative_err[~jnp.isnan(_data_flux)]
_img_date = jnp.array(img_date[~np.isnan(time_from_T0)] )
img_date = _img_date[~jnp.isnan(_data_flux)]

ramp_phase = jnp.array(ramp_phase[~np.isnan(time_from_T0)] )
breathing_phase = jnp.array(breathing_phase[~np.isnan(time_from_T0)] )
time_from_T0 = jnp.array(time_from_T0[~np.isnan(time_from_T0)] )

normalized_data_flux = data_flux/jnp.mean(data_flux)
normalized_relative_err = relative_err*normalized_data_flux

# plt.figure(figsize=(9,3) )
# plt.errorbar(time_from_T0,normalized_data_flux,yerr=normalized_relative_err,fmt='o',ms=1)
# plt.show()
# plt.clf()

### Make a dictionary of fiducial parameters

In [ ]:
# Make a dictionary of system and model parameters
default_params={

    # Ramp Model Parameters
    'r_1': 12.55,
    'r_2': -6.58,
    'r_3': -0.0101,
    #Breathing params
    'b_1':-0.06,
    'b_2':0.187,
    'b_3':-0.218,
    'b_4':0.093,

    # Other
    'beta':0.135,
    'baseline_slope':-0.03083,
    
    # Planet parameters
    'i_planet':89.5,
    'P_orb':8.463,
    't_0':0.000605,
    'rp_rstar': 0.04886,
    'a_rstar': 19.15,
    'ecc': 0.0,
    'u_1':0.23,
    'u_2':0.2,
    
    # Stellar parameters
    'log_g':4.5,
    'i_star':89.5,
    'P_rot':4.86,
    'Rs':0.8,
    'Ms':0.6,
    'metallicity':0.0,

    # Spot Parameters
    'T_phot': 3891,
    'T_spot': 3020,
}

### Make plot

In [ ]:
def calculate_model(parameter_dict,relative_flux):
    
    params = parameter_dict

    # Calculate systematics model
    ramp = ramp_model_jax(phase=ramp_phase,
                    r1=jnp.asarray(params.get('r_1',default_params['r_1']), dtype=jnp.float64),
                    r2=jnp.asarray(params.get('r_2',default_params['r_2']), dtype=jnp.float64),
                    r3=jnp.asarray(params.get('r_3',default_params['r_3']), dtype=jnp.float64) )
    breathing = breathing_model_jax(phase=breathing_phase,
                              b1=jnp.asarray(params.get('b_1',default_params['b_1']), dtype=jnp.float64),
                              b2=jnp.asarray(params.get('b_2',default_params['b_2']), dtype=jnp.float64),
                              b3=jnp.asarray(params.get('b_3',default_params['b_3']), dtype=jnp.float64),
                              b4=jnp.asarray(params.get('b_4',default_params['b_4']), dtype=jnp.float64))
    linear_baseline = linear_model_jax(x=time_from_T0,
                                        m=jnp.asarray(params.get('baseline_slope',default_params['baseline_slope']), dtype=jnp.float64),
                                        )
    _systematics = (breathing * ramp * linear_baseline)
    systematics = _systematics/jnp.mean(_systematics)
    
    # Update Planet Parameters
    planet_parameters = dict(
        inclination = jnp.radians(default_params['i_planet']),
        a = jnp.asarray(params.get('a_rstar',default_params['a_rstar']), dtype=jnp.float64),
        rp = jnp.asarray(params.get('rp_rstar',default_params['rp_rstar']), dtype=jnp.float64),
        period = jnp.asarray(params.get('P_orb',default_params['P_orb']), dtype=jnp.float64),
        t0 = jnp.asarray(params.get('t_0',default_params['t_0']), dtype=jnp.float64),
        ecc = jnp.asarray(params.get('ecc',default_params['ecc']), dtype=jnp.float64),
        u1 = jnp.asarray(params.get('u_1',default_params['u_1']), dtype=jnp.float64),
        u2 = jnp.asarray(params.get('u_2',default_params['u_2']), dtype=jnp.float64),
    )

    # Get spectra for the model temps
    Phot = get_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
                                    metallicity=0,
                                    data_wave = fleck_wavelengths)
    Cool = get_BTSettl_spectrum_jax(T=jnp.asarray(params.get('T_spot',default_params['T_spot']), dtype=jnp.float64),
                                    metallicity=0,
                                    data_wave = fleck_wavelengths)

    # Update stellar parameters
    active_star = ActiveStar(
        times = time_from_T0,
        inclination = jnp.radians(default_params['i_star']),
        T_eff=jnp.asarray(params.get('T_phot',default_params['T_phot']), dtype=jnp.float64),
        wavelength=Phot[0]*1e-6, # convert from micron to m
        phot=Phot[1],
        P_rot=default_params['P_rot']
    )
    
    active_star.spectrum = jnp.concatenate([
        jnp.vstack([Cool[1],Cool[1],Cool[1]]),])
    active_star.temperature = jnp.concatenate([
        jnp.array([params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot']),
                   params.get('T_spot', default_params['T_spot'])
                  ]),])
    active_star.lon = jnp.concatenate([
        jnp.array([0.1,0.2,0.2]),])
    active_star.lat = jnp.concatenate([
        jnp.array([0.1,0.2,0.3]),])
    active_star.rad = jnp.array([0,0,0])
    
    lc = active_star.transit_model(**planet_parameters)[0]
    lc_model = lc.T[lc_index] * systematics
    mean_per_wavelength = jnp.mean(lc_model)
    normalized_model = lc_model / mean_per_wavelength
    residuals = relative_flux - normalized_model
    de_trended_model = normalized_model / systematics
    de_trended_data = relative_flux/systematics

    return normalized_model, residuals, de_trended_model/jnp.mean(de_trended_model), de_trended_data/jnp.mean(de_trended_data)


def get_representative_sample_from_dict(posterior_dict_entry):
    """
    Works with the posterior dict structure you have.
    
    Parameters:
    -----------
    posterior_dict_entry : xarray Dataset
        The value from your posterior_samples dict (posterior_samples[lc_index])
    method : str
        'median' : Sample closest to medians (recommended)
    """
    # Extract the actual posterior dataset
    posterior_dataset = posterior_dict_entry
    
    # Get median values
    medians = posterior_dataset.median()
    
    # Get parameter names
    param_names = list(posterior_dataset.data_vars)
    
    # Prepare arrays
    all_samples = []
    median_values = []
    
    for param in param_names:
        # Get samples for this parameter (flatten chains and draws)
        samples = posterior_dataset[param].values.flatten()
        all_samples.append(samples)
        
        # Get median value
        median_values.append(float(medians[param]))
    
    # Convert to arrays
    param_matrix = np.column_stack(all_samples)  # n_samples x n_params
    median_array = np.array(median_values)       # n_params
    
    # Find sample closest to medians
    distances = np.linalg.norm(param_matrix - median_array, axis=1)
    closest_idx = np.argmin(distances)
    
    # Build representative parameter dict
    representative_params = {}
    for i, param in enumerate(param_names):
        representative_params[param] = param_matrix[closest_idx, i]
    
    return representative_params

def plot_final_lightcurve(jax_flux, jax_rel_err, time_from_T0, 
                           posterior_samples_dict, designation):
    
    fig, (ax_lc, ax_res) = plt.subplots(2, 1, figsize=(9, 5), sharex=True,height_ratios=[1,0.5])
    
    # Get median model for this light curve
    param_medians = get_representative_sample_from_dict(posterior_samples_dict[0])
    
    for name in param_medians.keys():
        print(f'{name} = {param_medians[str(name)]:.7f}')

    _, residuals, median_model, detrended_relative_flux = calculate_model(param_medians, jax_flux)

    # Plot light curve with offset on left panel
    ax_lc.errorbar(time_from_T0, detrended_relative_flux-0.00045,
                    yerr=jax_rel_err*detrended_relative_flux, fmt='o', alpha=0.7,
                    # color='blue',
                    markeredgecolor='k', markeredgewidth=0.5)
    
    # Plot median model on left panel
    ax_lc.plot(time_from_T0[0:20], median_model[0:20]-0.00045, '-', 
                color='k', linewidth=2, alpha=0.8)
    ax_lc.plot(time_from_T0[20:30], median_model[20:30]-0.00045, '-', 
                color='k', linewidth=2, alpha=0.8)
    ax_lc.plot(time_from_T0[30:], median_model[30:]-0.00045, '-', 
                color='k', linewidth=2, alpha=0.8)
    
    ax_res.errorbar(time_from_T0, residuals*1e6, 
                    yerr=jax_rel_err*1e6, fmt='o',
                    alpha=0.7)
    ax_res.axhline(0,color='k')
    
    # Set labels and formatting for LEFT panel
    ax_lc.set_ylabel('Relative Flux', fontsize=12)
    ax_res.set_ylabel('Residual Flux (ppm)', fontsize=12)
    ax_res.set_xlabel('Time from T0 (d)', fontsize=12)
    # ax_res.set_xlim(-0.09,0.09)
    fig.suptitle('F21/G141 White Light',fontsize=14)

    fig.tight_layout()
    
    # Save figure
    plt.savefig(f'{model_designation}_whitelight.png', dpi=400)
    plt.show()

In [ ]:
model_designation = 'F21_8000samples_8chains_whitelight_linear_2025_12_24'

posterior_samples = {}
result = az.InferenceData.from_netcdf(f'../../samples/{model_designation}')
posterior_samples[0] = result.posterior

# Create scaling factors array
scaling_factor = jnp.array([
    10**float(posterior_samples[0]['beta'].median().values)
])

jax_rel_err = scaling_factor * normalized_relative_err
jax_flux = normalized_data_flux

print(model_designation)

# Plot the mosaic
plot_final_lightcurve(jax_flux, jax_rel_err, time_from_T0,
    posterior_samples, model_designation)